# 臺北市河川淹水模擬：在 Colab 重跑全部資料、頁面與檢查

這本筆記本把 repo 裡的前處理（DEM 拼接、堤線、OSM、建物）、頁面組建、情境檢查與水利處積水模擬圖對照整個跑一遍，結果推回 GitHub。

- 執行階段：建議選 **CPU（高 RAM）**；不需要 GPU。全部七個河段約 2–3 小時（OSM 下載佔一大半，第二次起會用 Drive 快取）。
- 需要的秘密（左側 🔑 Secrets）：`GH_TOKEN`（GitHub fine-grained token，對 `xindian-flood-sim` 有 Contents 的讀寫權限），只有最後「推回 GitHub」那格會用到。
- 會掛載 Google Drive，把 `work/`（DEM 拼接結果、OSM 分塊、建物、測試頁）放在 `MyDrive/xindian-flood-sim-work` 當快取。

## 1. 掛載 Drive、取得 repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib
WORK = pathlib.Path('/content/drive/MyDrive/xindian-flood-sim-work'); WORK.mkdir(parents=True, exist_ok=True)
REPO = pathlib.Path('/content/xindian-flood-sim')
if REPO.exists():
    !cd {REPO} && git pull -q
else:
    !git clone -q https://github.com/henrychao521/xindian-flood-sim {REPO}
os.chdir(REPO)
if not (REPO / 'work').exists(): os.symlink(WORK, REPO / 'work')
!git log --oneline -1

## 2. 安裝工具（Python 套件、Playwright 的 Chromium、three.js）

In [ ]:
%pip install -q numpy scipy pyproj matplotlib
!apt-get -qq install -y fonts-wqy-zenhei > /dev/null
!node --version
!npm i -g --silent playwright@1.56.1 && npx -y playwright@1.56.1 install --with-deps chromium > /dev/null
!mkdir -p vendor && curl -sSL -o vendor/three.min.js https://cdn.jsdelivr.net/npm/three@0.128.0/build/three.min.js && curl -sSL -o vendor/OrbitControls.js https://cdn.jsdelivr.net/npm/three@0.128.0/examples/js/controls/OrbitControls.js && ls -la vendor

## 3. 建物資料（sheethub/tpe3d）與全市 DEM 拼接

In [ ]:
import pathlib
if not list(pathlib.Path('work/tpe3d').glob('*.json')):
    !git clone -q --depth 1 https://github.com/sheethub/tpe3d /content/tpe3d && mkdir -p work/tpe3d && cp /content/tpe3d/geojsons/*.json work/tpe3d/
!ls work/tpe3d | wc -l
if not pathlib.Path('work/taipei_dem20.npz').exists():
    !python3 scripts/mosaic.py
!ls -la work/taipei_dem20.npz

## 4. 重跑全部（資料 → 頁面 → 情境檢查 → 對照 → 總覽）

`scripts/run_checks.sh` 接受河段 id 當參數；想先試一個河段可以改成 `!bash scripts/run_checks.sh tamsui`。
OSM 分塊會從 api.openstreetmap.org 下載並快取在 `work/<id>/osm/`。

In [ ]:
!bash scripts/run_checks.sh 2>&1 | tail -60
!python3 scripts/results_table.py

## 5. 看結果，推回 GitHub（分支 `colab-results`，再在 GitHub 開 PR 合併）

In [ ]:
!git status --short | head -30
from google.colab import userdata
token = userdata.get('GH_TOKEN')
!git config user.name "henrychao521" && git config user.email "henrychao521@users.noreply.github.com"
!git checkout -q -B colab-results
!git add -A && git commit -q -m "Colab 重跑：資料、頁面、檢查結果、總覽" || echo "沒有變更"
!git push -q -f https://{token}@github.com/henrychao521/xindian-flood-sim colab-results && echo pushed